<a href="https://colab.research.google.com/github/ADITYA848-afk/ADITYA848-afk/blob/main/Touring_Package_Prediction_MLOps_COLAB_READY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Touring Package Prediction — End-to-End MLOps Project


**Goal:** Predict to identify customers likely to purchase the
**Touring Package**, enabling efficient marketing targeting.



In [ ]:
!pip -q install --upgrade pip
!pip -q install pandas numpy scikit-learn joblib huggingface_hub datasets streamlit pyyaml


## 1. Setup Repository Paths
If you cloned your GitHub repo, set `PROJECT_ROOT` to the repo folder.
Your dataset CSV must exist at: `data/raw_tourism.csv`.


In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd

# If you cloned your repo:
# !git clone https://github.com/<your-username>/touring-package-prediction-mlops.git
# PROJECT_ROOT = Path("touring-package-prediction-mlops")

PROJECT_ROOT = Path(".")  # change if needed
DATA_RAW_LOCAL = PROJECT_ROOT / "data" / "raw_tourism.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

for p in [PROCESSED_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("Raw data exists?", DATA_RAW_LOCAL.exists(), "|", DATA_RAW_LOCAL)


## 2. Hugging Face Configuration
Create these 3 **public** Hugging Face repos:
- Dataset: `datasets/<user>/touring-package-prediction`
- Model: `<user>/touring-package-prediction-model`
- Space: `spaces/<user>/touring-package-prediction-space`

Set `HF_TOKEN` (write token) in Colab environment variables for uploads.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download

HF_USERNAME = os.getenv("HF_USERNAME", "YOUR_HF_USERNAME")
HF_DATASET_REPO = os.getenv("HF_DATASET_REPO", "touring-package-prediction")
HF_MODEL_REPO = os.getenv("HF_MODEL_REPO", "touring-package-prediction-model")
HF_SPACE_REPO = os.getenv("HF_SPACE_REPO", "touring-package-prediction-space")
HF_TOKEN = os.getenv("HF_TOKEN", "")  # set via Colab secrets or os.environ

DATASET_REPO_ID = f"{HF_USERNAME}/{HF_DATASET_REPO}"
MODEL_REPO_ID = f"{HF_USERNAME}/{HF_MODEL_REPO}"
SPACE_REPO_ID = f"{HF_USERNAME}/{HF_SPACE_REPO}"

print("HF Dataset:", DATASET_REPO_ID)
print("HF Model:", MODEL_REPO_ID)
print("HF Space:", SPACE_REPO_ID)
print("HF_TOKEN set?", bool(HF_TOKEN))


## 3. Data Registration (HF Datasets)
Upload the raw dataset to Hugging Face Datasets as `raw.csv`.
If you haven’t created the dataset repo yet, create it first (public).


In [ ]:
api = HfApi(token=HF_TOKEN) if HF_TOKEN else None

if not DATA_RAW_LOCAL.exists():
    raise FileNotFoundError("raw_tourism.csv not found. Put your CSV at data/raw_tourism.csv in the repo.")

if api and HF_USERNAME != "YOUR_HF_USERNAME":
    api.upload_file(
        path_or_fileobj=str(DATA_RAW_LOCAL),
        path_in_repo="raw.csv",
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        commit_message="Add raw dataset for Touring Package prediction",
    )
    print("Uploaded raw.csv ->", DATASET_REPO_ID)
else:
    print("Skipped upload (set HF_TOKEN + HF_USERNAME).")


## 4. Load Dataset from Hugging Face (preferred) + Fallback to Local

In [ ]:
def load_raw():
    try:
        p = hf_hub_download(repo_id=DATASET_REPO_ID, filename="raw.csv", repo_type="dataset")
        print("Loaded raw from HF:", p)
        return pd.read_csv(p), "huggingface"
    except Exception as e:
        print("HF load failed; using local raw:", type(e).__name__)
        return pd.read_csv(DATA_RAW_LOCAL), "local"

df, src = load_raw()
print("Source:", src, "| shape:", df.shape)
df.head()


## 5. Data Cleaning & Preparation
**Cleaning steps**: drop index artifacts, remove duplicates, normalize categories, enforce target dtype.


In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in ["Unnamed: 0", "index"]:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)

    df = df.drop_duplicates()

    if "Occupation" in df.columns:
        df["Occupation"] = df["Occupation"].astype(str).str.strip().replace({"Free Lancer":"Freelancer"})
    for col in ["Gender","TypeofContact","MaritalStatus","ProductPitched","Designation"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    df["ProdTaken"] = df["ProdTaken"].astype(int)
    return df

df_clean = clean_dataframe(df)
print("Shape (clean):", df_clean.shape)
df_clean.isna().sum().sort_values(ascending=False).head(15)


## 6. Train-Test Split (save locally)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_clean, test_size=0.2, random_state=42, stratify=df_clean["ProdTaken"]
)

train_path = PROCESSED_DIR / "train.csv"
test_path = PROCESSED_DIR / "test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

train_df.shape, test_df.shape


## 7. Upload Train/Test back to Hugging Face (Datasets)

In [ ]:
if api and HF_USERNAME != "YOUR_HF_USERNAME":
    api.upload_file(str(train_path), "train.csv", repo_id=DATASET_REPO_ID, repo_type="dataset",
                    commit_message="Add train split")
    api.upload_file(str(test_path), "test.csv", repo_id=DATASET_REPO_ID, repo_type="dataset",
                    commit_message="Add test split")
    print("Uploaded train/test ->", DATASET_REPO_ID)
else:
    print("Skipped upload (set HF_TOKEN + HF_USERNAME).")


## 8. Model Building + Experimentation Tracking
We use an sklearn **Pipeline** (preprocessing + model) and tune hyperparameters.
We log the tuned parameters + metrics to `models/experiment.json`.
**Model used:** Decision Tree (rubric-allowed) with OneHotEncoding for categorical variables.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from joblib import dump

def load_split(name: str):
    try:
        p = hf_hub_download(repo_id=DATASET_REPO_ID, filename=name, repo_type="dataset")
        return pd.read_csv(p)
    except Exception:
        return pd.read_csv(PROCESSED_DIR / name)

train_df2 = load_split("train.csv")
test_df2 = load_split("test.csv")

drop_cols = [c for c in ["CustomerID"] if c in train_df2.columns]
X_train = train_df2.drop(columns=["ProdTaken"] + drop_cols)
y_train = train_df2["ProdTaken"].astype(int)
X_test = test_df2.drop(columns=["ProdTaken"] + drop_cols)
y_test = test_df2["ProdTaken"].astype(int)

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

pre = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

pipe = Pipeline([
    ("preprocess", pre),
    ("model", DecisionTreeClassifier(random_state=42, class_weight="balanced"))
])

param_dist = {
    "model__max_depth": [3, 5, 7, 9, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__criterion": ["gini", "entropy"],
}

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    random_state=42
)
search.fit(X_train, y_train)

best = search.best_estimator_
y_pred = best.predict(X_test)
proba = best.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, proba)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
}

experiment = {
    "best_params": search.best_params_,
    "best_cv_score_roc_auc": float(search.best_score_),
    "test_metrics": metrics,
    "feature_columns": X_train.columns.tolist(),
}

MODELS_DIR.mkdir(exist_ok=True)
dump(best, MODELS_DIR / "model.joblib")
(Path("models")/"experiment.json").write_text(json.dumps(experiment, indent=2))
(Path("models")/"feature_schema.json").write_text(json.dumps({"columns": X_train.columns.tolist()}, indent=2))

experiment


## 9. Register Best Model to Hugging Face Model Hub

In [ ]:
if api and HF_USERNAME != "YOUR_HF_USERNAME":
    api.upload_file("models/model.joblib", "model.joblib", repo_id=MODEL_REPO_ID, repo_type="model",
                    commit_message="Upload Touring Package model")
    api.upload_file("models/experiment.json", "metrics.json", repo_id=MODEL_REPO_ID, repo_type="model",
                    commit_message="Upload metrics")
    api.upload_file("models/feature_schema.json", "feature_schema.json", repo_id=MODEL_REPO_ID, repo_type="model",
                    commit_message="Upload schema")
    print("Uploaded model artifacts ->", MODEL_REPO_ID)
else:
    print("Skipped upload (set HF_TOKEN + HF_USERNAME).")


## 10. Deployment (Streamlit on Hugging Face Spaces)
Deployment files are provided in your repo:
- `deployment/app.py`
- `deployment/requirements.txt`
- `deployment/Dockerfile`
- `deployment/push_to_hf_space.py`

Run the push script after creating your Space repo.


In [ ]:
import subprocess, sys
push_script = PROJECT_ROOT / "deployment" / "push_to_hf_space.py"
print("Push script exists?", push_script.exists(), "|", push_script)

if push_script.exists() and HF_TOKEN and HF_USERNAME != "YOUR_HF_USERNAME":
    env = os.environ.copy()
    env["HF_TOKEN"] = HF_TOKEN
    env["HF_SPACE_REPO"] = SPACE_REPO_ID
    env["HF_MODEL_REPO"] = MODEL_REPO_ID
    subprocess.check_call([sys.executable, str(push_script)], env=env)
else:
    print("Skipped space push (ensure deployment files exist + set HF_TOKEN + HF_USERNAME).")


## 11. CI/CD with GitHub Actions
Workflow file: `.github/workflows/pipeline.yml` automates the end-to-end pipeline.
Add GitHub Secrets: `HF_TOKEN`, `HF_USERNAME`, `HF_MODEL_REPO`, `HF_SPACE_REPO`.


## 12. Final Outputs (MANDATORY)
Before exporting to HTML, paste these links and add screenshots:

- **GitHub Repo:** `https://github.com/<your-username>/touring-package-prediction-mlops`
- **HF Dataset:** `https://huggingface.co/datasets/<your-username>/touring-package-prediction`
- **HF Model:** `https://huggingface.co/<your-username>/touring-package-prediction-model`
- **HF Space:** `https://huggingface.co/spaces/<your-username>/touring-package-prediction-space`

**Screenshots to add:**
1) GitHub Actions successful run
2) HF Space Streamlit app running
